## Import thư viện và tạo SparkSession

In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, isnan, to_timestamp, date_format, hour,
    year, month, dayofmonth, dayofweek,
    when, lit, trim, upper, regexp_replace, add_months, concat
)
from pyspark.sql.types import IntegerType, DoubleType, StringType

# Ép Spark chạy dưới quyền root để HDFS cho phép ghi dữ liệu
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName("Retail_Bronze_To_Silver_Batch") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")

print("SparkSession đã khởi tạo thành công.")

SparkSession đã khởi tạo thành công.


## Khai báo đường dẫn Bronze và Silver

In [2]:
BRONZE_PATH = "hdfs://namenode:9000/data/bronze/retail/transactions"
SILVER_PATH = "hdfs://namenode:9000/data/silver/retail/transactions_cleaned"

print("Bronze path:", BRONZE_PATH)
print("Silver path:", SILVER_PATH)

Bronze path: hdfs://namenode:9000/data/bronze/retail/transactions
Silver path: hdfs://namenode:9000/data/silver/retail/transactions_cleaned


## Đọc dữ liệu từ Bronze Layer

In [3]:
try:
    bronze_df = spark.read.parquet(BRONZE_PATH)
    print("Đã đọc dữ liệu từ Bronze Layer thành công.")
    print(f"Số dòng Bronze: {bronze_df.count()}")
except Exception as e:
    print(f"Lỗi khi đọc dữ liệu từ Bronze Layer: {e}")
    raise e

bronze_df.printSchema()
bronze_df.show(10, truncate=False)

Đã đọc dữ liệu từ Bronze Layer thành công.
Số dòng Bronze: 2758005
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- batch_id: integer (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |batch_id|
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850.0   |United Kingdom|0       |
|536365   |71053    |WHITE METAL LANTERN       

## Kiểm tra dữ liệu lỗi trước khi clean

In [4]:
total_rows = bronze_df.count()

null_customer_count = bronze_df.filter(
    col("CustomerID").isNull() | isnan(col("CustomerID"))
).count()

bad_quantity_count = bronze_df.filter(
    col("Quantity").isNull() | (col("Quantity") <= 0)
).count()

bad_unitprice_count = bronze_df.filter(
    col("UnitPrice").isNull() | isnan(col("UnitPrice")) | (col("UnitPrice") <= 0)
).count()

null_invoice_date_count = bronze_df.filter(
    col("InvoiceDate").isNull()
).count()

print("===== BRONZE DATA QUALITY CHECK =====")
print(f"Tổng số dòng Bronze: {total_rows}")
print(f"Số dòng CustomerID null/NaN: {null_customer_count}")
print(f"Số dòng Quantity <= 0 hoặc null: {bad_quantity_count}")
print(f"Số dòng UnitPrice <= 0/null/NaN: {bad_unitprice_count}")
print(f"Số dòng InvoiceDate null: {null_invoice_date_count}")

===== BRONZE DATA QUALITY CHECK =====
Tổng số dòng Bronze: 2758005
Số dòng CustomerID null/NaN: 737929
Số dòng Quantity <= 0 hoặc null: 54612
Số dòng UnitPrice <= 0/null/NaN: 13434
Số dòng InvoiceDate null: 0


## Clean và chuẩn hóa dữ liệu
    CustomerID null → -1
    Country null → Unknown

In [5]:
silver_df = bronze_df

# Ép kiểu dữ liệu quan trọng
silver_df = silver_df.withColumn("Quantity", col("Quantity").cast(IntegerType()))
silver_df = silver_df.withColumn("UnitPrice", col("UnitPrice").cast(DoubleType()))
silver_df = silver_df.withColumn("CustomerID", col("CustomerID").cast(DoubleType()))

# Loại các giao dịch không hợp lệ
# Quantity <= 0 thường là đơn hủy hoặc dòng lỗi
# UnitPrice <= 0 không tạo doanh thu hợp lệ
silver_df = silver_df.filter(
    col("Quantity").isNotNull() &
    (col("Quantity") > 0) &
    col("UnitPrice").isNotNull() &
    (~isnan(col("UnitPrice"))) &
    (col("UnitPrice") > 0)
)

# Parse InvoiceDate sang Timestamp
silver_df = silver_df.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

# Loại dòng không parse được ngày
silver_df = silver_df.filter(col("InvoiceDate").isNotNull())

# Xử lý CustomerID null
silver_df = silver_df.withColumn(
    "CustomerID",
    when(
        col("CustomerID").isNull() | isnan(col("CustomerID")),
        lit(-1)
    ).otherwise(col("CustomerID").cast(IntegerType()))
)

# Xử lý Country null
silver_df = silver_df.withColumn(
    "Country",
    when(
        col("Country").isNull() | (trim(col("Country")) == ""),
        lit("Unknown")
    ).otherwise(trim(col("Country")))
)

# Chuẩn hóa Description
silver_df = silver_df.withColumn(
    "Description",
    when(
        col("Description").isNull() | (trim(col("Description")) == ""),
        lit("Unknown Product")
    ).otherwise(trim(col("Description")))
)

# Chuẩn hóa StockCode
silver_df = silver_df.withColumn(
    "StockCode",
    trim(col("StockCode").cast(StringType()))
)

# Chuẩn hóa InvoiceNo
silver_df = silver_df.withColumn(
    "InvoiceNo",
    trim(col("InvoiceNo").cast(StringType()))
)

# Tạo các cột phân tích
silver_df = silver_df.withColumn(
    "DateKey",
    date_format(col("InvoiceDate"), "yyyyMMdd").cast(IntegerType())
)

silver_df = silver_df.withColumn(
    "Hour",
    hour(col("InvoiceDate"))
)

silver_df = silver_df.withColumn(
    "Year",
    year(col("InvoiceDate"))
)

silver_df = silver_df.withColumn(
    "Month",
    month(col("InvoiceDate"))
)

silver_df = silver_df.withColumn(
    "Day",
    dayofmonth(col("InvoiceDate"))
)

silver_df = silver_df.withColumn(
    "DayOfWeek",
    dayofweek(col("InvoiceDate"))
)

silver_df = silver_df.withColumn(
    "TotalPrice",
    col("Quantity") * col("UnitPrice")
)

# Xóa duplicate ở mức record giao dịch
silver_df = silver_df.dropDuplicates([
    "InvoiceNo",
    "StockCode",
    "CustomerID",
    "InvoiceDate",
    "Quantity",
    "UnitPrice"
])

print("Đã clean và chuẩn hóa dữ liệu.")
print(f"Số dòng Silver sau khi clean: {silver_df.count()}")

Đã clean và chuẩn hóa dữ liệu.
Số dòng Silver sau khi clean: 524876


## ENRICH SILVER DATA VỚI WEATHER VÀ HOLIDAYS

In [6]:
WEATHER_PATH = "hdfs://namenode:9000/data/bronze/weather_optimized"
HOLIDAYS_PATH = "hdfs://namenode:9000/data/bronze/holidays_optimized"

print("Đang đọc dữ liệu Weather từ HDFS...")
weather_df = spark.read.parquet(WEATHER_PATH)

print("Đang đọc dữ liệu Holidays từ HDFS...")
holidays_df = spark.read.parquet(HOLIDAYS_PATH)

# Ép DateKey về integer để join đúng kiểu
weather_df = weather_df.withColumn(
    "DateKey",
    col("DateKey").cast(IntegerType())
).dropDuplicates(["DateKey"])

holidays_df = holidays_df.withColumn(
    "DateKey",
    col("DateKey").cast(IntegerType())
).dropDuplicates(["DateKey"])

print("Weather schema:")
weather_df.printSchema()

print("Holidays schema:")
holidays_df.printSchema()

# Join transaction cleaned với weather
silver_df = silver_df.join(
    weather_df,
    on="DateKey",
    how="left"
)

# Join tiếp với holidays
silver_df = silver_df.join(
    holidays_df,
    on="DateKey",
    how="left"
)

# Fill null nếu ngày đó không có weather/holiday
silver_df = silver_df.fillna({
    "IsHoliday": False,
    "HolidayName": "Regular Day",
    "Temperature": 0.0,
    "Rainfall": 0.0,
    "Snowfall": 0.0
})

print("Đã enrich Silver với Weather và Holidays.")
silver_df.show(10, truncate=False)

Đang đọc dữ liệu Weather từ HDFS...
Đang đọc dữ liệu Holidays từ HDFS...
Weather schema:
root
 |-- DateKey: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)

Holidays schema:
root
 |-- DateKey: integer (nullable = true)
 |-- HolidayName: string (nullable = true)
 |-- IsHoliday: boolean (nullable = true)

Đã enrich Silver với Weather và Holidays.
+--------+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+--------+----+----+-----+---+---------+----------+-----------+--------+--------+-----------+---------+
|DateKey |InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |batch_id|Hour|Year|Month|Day|DayOfWeek|TotalPrice|Temperature|Rainfall|Snowfall|HolidayName|IsHoliday|
+--------+---------+---------+-----------------------------------+--------+---------

## Kiểm tra dữ liệu Silver

In [7]:
silver_df.printSchema()
silver_df.show(10, truncate=False)

root
 |-- DateKey: integer (nullable = true)
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- Temperature: double (nullable = false)
 |-- Rainfall: double (nullable = false)
 |-- Snowfall: double (nullable = false)
 |-- HolidayName: string (nullable = false)
 |-- IsHoliday: boolean (nullable = false)

+--------+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+--

In [8]:

print("===== SILVER DATA QUALITY CHECK =====")

print("Tổng số dòng Silver:", silver_df.count())

print("Số dòng CustomerID null:", silver_df.filter(col("CustomerID").isNull()).count())

print("Số dòng Quantity <= 0:", silver_df.filter(col("Quantity") <= 0).count())

print("Số dòng UnitPrice <= 0:", silver_df.filter(col("UnitPrice") <= 0).count())

print("Số dòng InvoiceDate null:", silver_df.filter(col("InvoiceDate").isNull()).count())

print("Số dòng DateKey null:", silver_df.filter(col("DateKey").isNull()).count())

print("Số dòng Hour null:", silver_df.filter(col("Hour").isNull()).count())

===== SILVER DATA QUALITY CHECK =====
Tổng số dòng Silver: 524876
Số dòng CustomerID null: 0
Số dòng Quantity <= 0: 0
Số dòng UnitPrice <= 0: 0
Số dòng InvoiceDate null: 0
Số dòng DateKey null: 0
Số dòng Hour null: 0


In [9]:
silver_df.groupBy("Hour") \
    .count() \
    .orderBy(col("count"), ascending=False) \
    .show(24, truncate=False)

+----+-----+
|Hour|count|
+----+-----+
|12  |75986|
|15  |75665|
|13  |69992|
|14  |65057|
|11  |55419|
|16  |52992|
|10  |47597|
|9   |33684|
|17  |27426|
|8   |8797 |
|18  |7676 |
|19  |3427 |
|20  |778  |
|7   |379  |
|6   |1    |
+----+-----+



## Ghi dữ liệu sang Silver Layer

In [10]:
try:
    silver_df.write \
        .mode("overwrite") \
        .parquet(SILVER_PATH)

    print(f"Đã ghi dữ liệu sạch vào Silver Layer: {SILVER_PATH}")

except Exception as e:
    print(f"Lỗi khi ghi dữ liệu vào Silver Layer: {e}")
    raise e

Đã ghi dữ liệu sạch vào Silver Layer: hdfs://namenode:9000/data/silver/retail/transactions_cleaned


## Đọc lại Silver để kiểm tra

In [11]:
silver_check_df = spark.read.parquet(SILVER_PATH)

print("Đọc lại Silver Layer thành công.")
print(f"Số dòng trong Silver Layer: {silver_check_df.count()}")

silver_check_df.printSchema()
silver_check_df.show(10, truncate=False)

Đọc lại Silver Layer thành công.
Số dòng trong Silver Layer: 524876
root
 |-- DateKey: integer (nullable = true)
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)
 |-- HolidayName: string (nullable = true)
 |-- IsHoliday: boolean (nullable = true)

+--------+---------+---------+-----------------------------------+----